In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import re

from collections import Counter
from google.colab import files

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay


In [ ]:
from google.colab import files

uploaded = files.upload()


## Analytical Engineer


In [ ]:
df=pd.read_csv("C:/Users/RAKSHITH.LAPTOP-FM0GREEU/Downloads/processed_data.csv")
display(df.head())

In [ ]:

# Split the dataset into 80% training and 20% testing
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Shape of training data: {train_df.shape}")
print(f"Shape of testing data: {test_df.shape}")

display(train_df.head())
display(test_df.head())

In [ ]:

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # You can adjust max_features as needed

# Fit the vectorizer on the training data 'review' column and transform it
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['review'])

# Transform the testing data 'review' column
X_test_tfidf = tfidf_vectorizer.transform(test_df['review'])

# Display the shapes of the resulting TF-IDF matrices
print(f"Shape of TF-IDF matrix for training data: {X_train_tfidf.shape}")
print(f"Shape of TF-IDF matrix for testing data: {X_test_tfidf.shape}")

In [ ]:

# Define target variables
y_train = train_df['sentiment']
y_test = test_df['sentiment']

# Initialize and train a Logistic Regression model
lr_model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is good for small datasets and binary classification
lr_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred_lr = lr_model.predict(X_test_tfidf)

# Evaluate the model
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

In [ ]:

# Initialize and train a Linear SVM model
svm_model = LinearSVC(random_state=42, dual=True) # dual=True is default and usually good for n_samples > n_features
svm_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred_svm = svm_model.predict(X_test_tfidf)

# Evaluate the model
print("Linear SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

In [ ]:

# Initialize and train a Multinomial Naive Bayes model
mnb_model = MultinomialNB()
mnb_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred_mnb = mnb_model.predict(X_test_tfidf)

# Evaluate the model
print("Multinomial Naive Bayes Classification Report:")
print(classification_report(y_test, y_pred_mnb))

In [ ]:

# Define the features (X) and target (y) from the original DataFrame
X = df['review']
y = df['sentiment']

# Initialize Stratified K-Fold for 5 splits
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Performing 5-fold cross-validation...")

# --- Logistic Regression with Pipeline ---
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)), # Use same max_features as before
    ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
])

lr_cv_scores = cross_val_score(lr_pipeline, X, y, cv=stratified_kfold, scoring='accuracy', n_jobs=-1)
print(f"\nLogistic Regression (5-fold CV) Mean Accuracy: {lr_cv_scores.mean():.4f}")
print(f"All LR CV Scores: {lr_cv_scores}")

# --- Linear SVM with Pipeline ---
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('classifier', LinearSVC(random_state=42, dual=True))
])

svm_cv_scores = cross_val_score(svm_pipeline, X, y, cv=stratified_kfold, scoring='accuracy', n_jobs=-1)
print(f"\nLinear SVM (5-fold CV) Mean Accuracy: {svm_cv_scores.mean():.4f}")
print(f"All SVM CV Scores: {svm_cv_scores}")

# --- Multinomial Naive Bayes with Pipeline ---
mnb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('classifier', MultinomialNB())
])

mnb_cv_scores = cross_val_score(mnb_pipeline, X, y, cv=stratified_kfold, scoring='accuracy', n_jobs=-1)
print(f"\nMultinomial Naive Bayes (5-fold CV) Mean Accuracy: {mnb_cv_scores.mean():.4f}")
print(f"All MNB CV Scores: {mnb_cv_scores}")

In [ ]:

print("Performing hyperparameter tuning for Logistic Regression...")

# Define a pipeline for Logistic Regression
# We'll use a new TfidfVectorizer instance within the pipeline to avoid data leakage
# if we were to further split data or use this pipeline on raw text.
# However, for GridSearchCV, it's often more efficient to define the pipeline directly.
lr_tuning_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('classifier', LogisticRegression(random_state=42))
])

# Define the parameter grid for Logistic Regression
param_grid_lr = {
    'classifier__C': [0.1, 1, 10], # Inverse of regularization strength; smaller values specify stronger regularization
    'classifier__solver': ['liblinear', 'saga'] # Solvers that work well with L1/L2 regularization
}

# Initialize GridSearchCV
# We'll use the same stratified_kfold from before for consistent CV splits
grid_search_lr = GridSearchCV(lr_tuning_pipeline, param_grid_lr, cv=stratified_kfold, scoring='accuracy', n_jobs=-1, verbose=1)

# Fit the grid search to the training data (original X and y for CV)
grid_search_lr.fit(X, y)

# Print the best parameters and best score
print(f"\nBest parameters for Logistic Regression: {grid_search_lr.best_params_}")
print(f"Best cross-validation accuracy for Logistic Regression: {grid_search_lr.best_score_:.4f}")

In [ ]:

# Calculate metrics for Logistic Regression
lr_acc = accuracy_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr, average='macro')

# Calculate metrics for Linear SVM
svm_acc = accuracy_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm, average='macro')

# Calculate metrics for Multinomial Naive Bayes
mnb_acc = accuracy_score(y_test, y_pred_mnb)
mnb_f1 = f1_score(y_test, y_pred_mnb, average='macro')

# Create a comparison DataFrame
comparison_data = {
    'Model': ['Logistic Regression', 'Linear SVM', 'Multinomial Naive Bayes'],
    'Accuracy': [lr_acc, svm_acc, mnb_acc],
    'Macro F1-Score': [lr_f1, svm_f1, mnb_f1]
}

comparison_df = pd.DataFrame(comparison_data)

# Display the comparison table
display(comparison_df.style.format({
    'Accuracy': '{:.4%}',
    'Macro F1-Score': '{:.4%}'
}))

In [ ]:

# Set up the plotting area with 3 subplots side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot Logistic Regression Confusion Matrix ---
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Negative (0)', 'Positive (1)'])
disp_lr.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Logistic Regression')

# --- Plot Linear SVM Confusion Matrix ---
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=['Negative (0)', 'Positive (1)'])
disp_svm.plot(ax=axes[1], cmap='Blues', values_format='d')
axes[1].set_title('Linear SVM')

# --- Plot Multinomial Naive Bayes Confusion Matrix ---
cm_mnb = confusion_matrix(y_test, y_pred_mnb)
disp_mnb = ConfusionMatrixDisplay(confusion_matrix=cm_mnb, display_labels=['Negative (0)', 'Positive (1)'])
disp_mnb.plot(ax=axes[2], cmap='Blues', values_format='d')
axes[2].set_title('Multinomial Naive Bayes')

plt.tight_layout()
plt.show()

In [ ]:
# Input and target
X = df["review"]
y = df["sentiment"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert text into numerical TF-IDF features
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


# Create the Logistic Regression model
model = LogisticRegression(
    random_state=42,
    solver="liblinear"
)

# Train the model
model.fit(X_train_tfidf, y_train)

print("Logistic Regression model trained successfully.")

In [ ]:
# Generate predictions
prediction = model.predict(X_test_tfidf)

# Generate probability of positive class
probability_positive = model.predict_proba(X_test_tfidf)[:, 1]

print("Predictions generated successfully.")
print("Number of predictions:", len(prediction))

In [ ]:
# Display probabilities for both negative and positive classes
model.predict_proba(X_test_tfidf)

In [ ]:
# Import pandas for creating the analytical DataFrame

# Combine actual sentiment, model prediction and positive probability
ae_df = pd.DataFrame({
    "actual_sentiment": y_test.values,       # Actual sentiment from the test dataset
    "prediction": prediction,                # Sentiment predicted by the model
    "probability_positive": probability_positive  # Model's positive probability
})

# Display the first few rows of the analytical dataset
print(ae_df.head())

In [ ]:
prediction = model.predict(X_test_tfidf)
probability_positive = model.predict_proba(X_test_tfidf)[:, 1]
ae_df.head()

In [ ]:
# Check what columns currently exist in ae_df
print(ae_df.columns.tolist())

In [ ]:
# Import confusion matrix from Scikit-learn

# Import visualization libraries

# Compare actual sentiment with predicted sentiment
cm = confusion_matrix(
    ae_df["actual_sentiment"],
    ae_df["prediction"]
)

# Display the numerical confusion matrix
print("Confusion Matrix:")
print(cm)

# Visualize the confusion matrix as a heatmap
sns.heatmap(
    cm,
    annot=True,       # Display values inside the matrix
    fmt="d",          # Display values as integers
    cmap="Blues"      # Color style for visualization
)

# Label the axes
plt.xlabel("Predicted")
plt.ylabel("Actual")

# Give the chart a title
plt.title("Logistic Regression - Confusion Matrix")

# Display the chart
plt.show()

In [ ]:
# Extract the four values from the confusion matrix
TN, FP, FN, TP = cm.ravel()

# Display True Negative count
print("True Negative :", TN)

# Display False Positive count
print("False Positive:", FP)

# Display False Negative count
print("False Negative:", FN)

# Display True Positive count
print("True Positive :", TP)

In [ ]:
# Calculate Type I error rate using False Positives
type_1_error_rate = FP / (FP + TN)

# Calculate Type II error rate using False Negatives
type_2_error_rate = FN / (FN + TP)

# Display Type I error rate
print("Type I Error Rate :", type_1_error_rate)

# Display Type II error rate
print("Type II Error Rate:", type_2_error_rate)

In [ ]:
print("Type I Error Rate:",
      round(type_1_error_rate * 100, 2), "%")

print("Type II Error Rate:",
      round(type_2_error_rate * 100, 2), "%")

In [ ]:
# Assign a business cost of ₹5 to every False Positive
FP_cost = 5

# Assign a business cost of ₹10 to every False Negative
FN_cost = 10

# Calculate the total business cost of model errors
business_cost = (FP * FP_cost) + (FN * FN_cost)

# Display the total business cost
print("Business Cost: ₹", business_cost)

In [ ]:
# Define the thresholds to evaluate
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

results = []

# Loop through every selected threshold
for threshold in thresholds:

    # Generate predictions using the current threshold
    threshold_prediction = (
        ae_df["probability_positive"] >= threshold
    ).astype(int)

    # Calculate the confusion matrix
    cm = confusion_matrix(
        ae_df["actual_sentiment"],
        threshold_prediction
    )

    TN, FP, FN, TP = cm.ravel()

    # Calculate business cost
    business_cost = (FP * 5) + (FN * 10)

    results.append({
        "Threshold": threshold,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TP": TP,
        "Total_Cost": business_cost
    })

# Create results table
results_df = pd.DataFrame(results)

print(results_df)

In [ ]:
# Select minimum-cost threshold
best_threshold = results_df.loc[
    results_df["Total_Cost"].idxmin(),
    "Threshold"
]

print("Selected Business Threshold:", best_threshold)

# Final predictions
ae_df["final_prediction"] = (
    ae_df["probability_positive"] >= best_threshold
).astype(int)

print("\nFinal predictions:")
print(ae_df.head())

## Data Analyst (EDA)


In [ ]:
df = pd.read_csv('/content/compressed_data.csv')
df.head()

### 1. Dataset Understanding & Basic EDA

This section provides a high-level overview of the dataset, including its structure, data types, and initial descriptive statistics.

In [ ]:
print("--- df.info() ---")
df.info()
print("\n--- df.head() ---")
print(df.head())
print("\n--- df.describe() (for numerical columns) ---")
print(df.describe())

### 2. Target/Class Distribution Analysis

Here, we examine the distribution of the target variable (`sentiment`) to understand if there's any class imbalance.

In [ ]:

sns.set_style("whitegrid")

print("--- Sentiment Value Counts ---")
print(df['sentiment'].value_counts())
print("\n--- Sentiment Value Counts (Normalized) ---")
print(df['sentiment'].value_counts(normalize=True))

plt.figure(figsize=(6, 4))
sns.countplot(x='sentiment', data=df)
plt.title('Distribution of Sentiment')
plt.xlabel('Sentiment (0: Negative, 1: Positive)')
plt.ylabel('Count')
plt.show()

### 3. Missing-Value Analysis

This step checks for any missing values in the dataset, which can impact data quality and analysis.

In [ ]:
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing Percent': missing_percent})
print(missing_df)
print(f"Total missing values in the dataset: {df.isnull().sum().sum()}")

### 4. Duplicate Analysis

We identify and count any duplicate rows in the dataset to ensure data uniqueness.

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

### 5 & 6. Review-Length and Distribution Analysis

This section analyzes the length of reviews (both character and token counts) and visualizes their distributions.

In [ ]:

# Ensure 'review_length' (character count) is available
if 'review_length' not in df.columns:
    df['review_length'] = df['review'].apply(len)

# Ensure 'token_length' is available (length of the cleaned token list)
if 'token_length' not in df.columns:
    df['token_length'] = df['review_tokens_cleaned'].apply(len)

print("--- Descriptive Statistics for Review Lengths ---")
print(df[['review_length', 'token_length']].describe())

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['review_length'], bins=50, kde=True)
plt.title('Distribution of Review Character Length')
plt.xlabel('Character Length')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.histplot(df['token_length'], bins=50, kde=True)
plt.title('Distribution of Review Token Length')
plt.xlabel('Token Length')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

### 7. Outlier/Anomaly Detection (Review Length)

We use box plots and the IQR method to identify potential outliers in review lengths.

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(y=df['review_length'])
plt.title('Box Plot of Review Character Length')
plt.ylabel('Character Length')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['token_length'])
plt.title('Box Plot of Review Token Length')
plt.ylabel('Token Length')
plt.tight_layout()
plt.show()

# Identify outliers using IQR method for token_length
Q1_tokens = df['token_length'].quantile(0.25)
Q3_tokens = df['token_length'].quantile(0.75)
IQR_tokens = Q3_tokens - Q1_tokens
lower_bound_tokens = Q1_tokens - 1.5 * IQR_tokens
upper_bound_tokens = Q3_tokens + 1.5 * IQR_tokens

outliers_tokens = df[(df['token_length'] < lower_bound_tokens) | (df['token_length'] > upper_bound_tokens)]
print(f"Number of token length outliers: {len(outliers_tokens)}")
print(f"Percentage of token length outliers: {len(outliers_tokens) / len(df) * 100:.2f}%")

### 8. Outlier Removal

Based on the IQR method for `token_length`, we will remove the identified outliers from the dataset to ensure more robust analysis in subsequent steps.

In [ ]:
# Remove outliers based on token_length using the calculated IQR bounds
df = df[(df['token_length'] >= lower_bound_tokens) & (df['token_length'] <= upper_bound_tokens)]
print(f"Number of rows after outlier removal: {len(df)}")

### 9. Positive vs Negative Comparison (Review Length)

We compare the distribution of review lengths between positive and negative sentiments, now with outliers removed.

In [ ]:

# df is assumed to be loaded and cleaned from previous cells

print("--- Descriptive Statistics of Token Length by Sentiment (Outliers Cleared) ---")
print(df.groupby('sentiment')['token_length'].describe())

plt.figure(figsize=(7, 5))
sns.boxplot(x='sentiment', y='token_length', data=df)
plt.title('Token Length Distribution by Sentiment (Outliers Cleared)')
plt.xlabel('Sentiment (0: Negative, 1: Positive)')
plt.ylabel('Token Length')
plt.show()

### 9. Word-Frequency Analysis

This analysis identifies the most frequent words in the entire dataset, as well as separately for positive and negative reviews.

In [ ]:

# df is assumed to be loaded and cleaned from previous cells

# Flatten the list of tokens and count frequencies
all_words = pd.Series(list(itertools.chain.from_iterable(df['review_tokens_cleaned'])))
positive_words = pd.Series(list(itertools.chain.from_iterable(df[df['sentiment'] == 1]['review_tokens_cleaned'])))
negative_words = pd.Series(list(itertools.chain.from_iterable(df[df['sentiment'] == 0]['review_tokens_cleaned'])))

common_all = Counter(all_words).most_common(20)
common_positive = Counter(positive_words).most_common(20)
common_negative = Counter(negative_words).most_common(20)

print("--- 20 Most Common Words (Overall - Outliers Cleared) ---")
print(common_all)
print("\n--- 20 Most Common Words (Positive Reviews - Outliers Cleared) ---")
print(common_positive)
print("\n--- 20 Most Common Words (Negative Reviews - Outliers Cleared) ---")
print(common_negative)

def plot_word_frequencies(word_counts, title):
    words, counts = zip(*word_counts)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(counts), y=list(words), palette='viridis', hue=list(words), legend=False)
    plt.title(title)
    plt.xlabel('Frequency')
    plt.ylabel('Word')
    plt.tight_layout()
    plt.show()

plot_word_frequencies(common_all, 'Top 20 Most Common Words (Overall - Outliers Cleared)')
plot_word_frequencies(common_positive, 'Top 20 Most Common Words (Positive Reviews - Outliers Cleared)')
plot_word_frequencies(common_negative, 'Top 20 Most Common Words (Negative Reviews - Outliers Cleared)')

### 10. N-gram Analysis (Bigrams and Trigrams)

This section explores the most frequent sequences of two (bigrams) and three (trigrams) words, providing more contextual insights.

In [ ]:

# df is assumed to be loaded and cleaned from previous cells

def generate_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))

all_bigrams = list(itertools.chain.from_iterable(df['review_tokens_cleaned'].apply(lambda x: generate_ngrams(x, 2))))
all_trigrams = list(itertools.chain.from_iterable(df['review_tokens_cleaned'].apply(lambda x: generate_ngrams(x, 3))))

top_20_bigrams = Counter(all_bigrams).most_common(20)
top_20_trigrams = Counter(all_trigrams).most_common(20)

print("--- Top 20 Bigrams (Outliers Cleared) ---")
print(top_20_bigrams)
print("\n--- Top 20 Trigrams (Outliers Cleared) ---")
print(top_20_trigrams)

def plot_ngram_frequencies(ngram_counts, title):
    ngrams = [" ".join(ngram) for ngram, count in ngram_counts]
    counts = [count for ngram, count in ngram_counts]
    plt.figure(figsize=(10, 6))
    sns.barplot(x=counts, y=ngrams, palette='magma')
    plt.title(title)
    plt.xlabel('Frequency')
    plt.ylabel('N-gram')
    plt.tight_layout()
    plt.show()

plot_ngram_frequencies(top_20_bigrams, 'Top 20 Bigrams (Outliers Cleared)')
plot_ngram_frequencies(top_20_trigrams, 'Top 20 Trigrams (Outliers Cleared)')

### 11. Feature Relationship/Correlation Analysis (e.g., Review Length vs. Sentiment) (Outliers Cleared)

We examine potential relationships between different features, such as how review length correlates with sentiment, now with outliers removed.

In [ ]:

# df is assumed to be loaded and cleaned from previous cells

print("--- Mean Token Length per Sentiment (Outliers Cleared) ---")
print(df.groupby('sentiment')['token_length'].mean())

# Although sentiment is categorical, we can visualize the relationship
plt.figure(figsize=(7, 5))
sns.barplot(x='sentiment', y='token_length', data=df)
plt.title('Bar Plot of Token Length by Sentiment (Outliers Cleared)')
plt.xlabel('Sentiment (0: Negative, 1: Positive)')
plt.ylabel('Token Length')
plt.show()

### 12. Statistical Analysis (Summary of Key Statistics) (Outliers Cleared)

This provides a concise summary of key statistical measures for relevant numerical features, now with outliers removed.

In [ ]:

# df is assumed to be loaded and cleaned from previous cells

print("--- Summary Statistics for Token Length (Outliers Cleared) ---")
print(df['token_length'].describe())

print("\n--- Summary Statistics for Review Character Length (Outliers Cleared) ---")
print(df['review_length'].describe())

### 13. Token Cleaning: Removing Non-Alphabetic Tokens

To ensure our word and n-gram analyses are based on meaningful words, we will further clean the `review_tokens_cleaned` column by removing any tokens that are not purely alphabetic and converting all tokens to lowercase. This will eliminate punctuation, numbers, and single characters that were skewing our previous frequency counts.

In [ ]:

def clean_tokens(tokens):
    cleaned = []
    for token in tokens:
        # Convert to lowercase and remove non-alphabetic characters
        # Keep apostrophes if part of a word (e.g., 'don't', 'it's')
        processed_token = re.sub(r'[^a-zÀ-ſĀ-ſЀ-ӿ]+', '', token.lower())
        # Remove empty strings that might result from cleaning
        if processed_token and len(processed_token) > 1: # Also remove single characters
            cleaned.append(processed_token)
    return cleaned

# Apply the cleaning function to the review_tokens_cleaned column
df['review_tokens_cleaned'] = df['review_tokens_cleaned'].apply(clean_tokens)

print("--- Sample of cleaned tokens (first 5 reviews) ---")
for i, tokens in enumerate(df['review_tokens_cleaned'].head()):
    print(f"Review {i}: {tokens}")
    if i >= 4: # Limit to first 5 for brevity
        break

## Data Scientist


In [ ]:

# Load the uploaded dataset
df = pd.read_csv('compressed_data.csv')

# Display the original shape and first few rows
print(f"Original dataset shape: {df.shape}")
display(df.head())

# Split the dataset into train and test sets (70% train, 30% test)
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

# Display the shapes of the split datasets
print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

In [ ]:

# 1. Initialize the TF-IDF Vectorizer
# We limit max_features to 10,000 to keep the matrix size manageable
tfidf_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')

# 2. Fit and transform the training text data
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['review'])

# 3. Transform the testing text data
X_test_tfidf = tfidf_vectorizer.transform(test_df['review'])

# Print the shape of the resulting TF-IDF matrices
print(f"TF-IDF Train Matrix Shape: {X_train_tfidf.shape}")
print(f"TF-IDF Test Matrix Shape: {X_test_tfidf.shape}")

# 4. View some of the feature names (vocabulary)
feature_names = tfidf_vectorizer.get_feature_names_out()
print("\nFirst 20 feature names:")
print(feature_names[:20])

# 5. Represent a portion of the TF-IDF vector for the first document as a DataFrame
first_doc_vector = X_train_tfidf[0].toarray()
tfidf_representation = pd.DataFrame(first_doc_vector, columns=feature_names)

# Filter out words with 0 TF-IDF weight to see the most important words in the first review
non_zero_tfidf = tfidf_representation.T[tfidf_representation.T[0] > 0]
print("\nNon-zero TF-IDF values for the first document:")
display(non_zero_tfidf.sort_values(by=0, ascending=False).head(10))

In [ ]:

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=10000)

# Vectorize the text data (using the raw 'review' column)
X_train = vectorizer.fit_transform(train_df['review'])
X_test = vectorizer.transform(test_df['review'])

y_train = train_df['sentiment']
y_test = test_df['sentiment']

# Initialize and train Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# Make predictions
y_pred = lr_model.predict(X_test)

# Evaluate the model
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:

# Initialize Linear SVM classifier
# We use LinearSVC as it is optimized specifically for linear kernels and handles sparse matrices well
svm_model = LinearSVC(random_state=42, max_iter=2000)

# Train the SVM model on the TF-IDF vectorized training data
svm_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_svm = svm_model.predict(X_test)

# Evaluate the SVM model
print(f"Linear SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}\n")
print("Classification Report for Linear SVM:")
print(classification_report(y_test, y_pred_svm))

In [ ]:

# Initialize Multinomial Naive Bayes classifier
mnb_model = MultinomialNB()

# Train the Multinomial Naive Bayes model on the TF-IDF vectorized training data
mnb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_mnb = mnb_model.predict(X_test)

# Evaluate the Multinomial Naive Bayes model
print(f"Multinomial Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_mnb):.4f}\n")
print("Classification Report for Multinomial Naive Bayes:")
print(classification_report(y_test, y_pred_mnb))

In [ ]:

# Define X and y for cross-validation
# Using the TF-IDF vectorized training data (X_train) and corresponding labels (y_train)
X = X_train
y = y_train

# Initialize Multinomial Naive Bayes classifier (re-using the previously defined mnb_model)
# If you wanted to run cross-validation on a fresh model, you would re-initialize here:
# mnb_model_cv = MultinomialNB()

# Set up 5-fold stratified cross-validation
# StratifiedKFold ensures that each fold has the same proportion of classes as the entire dataset.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform cross-validation
cv_scores = cross_val_score(mnb_model, X, y, cv=skf, scoring='accuracy', n_jobs=-1)

# Print the cross-validation scores
print(f"Individual cross-validation accuracy scores: {cv_scores}")
print(f"Mean cross-validation accuracy: {cv_scores.mean():.4f}")
print(f"Standard deviation of cross-validation accuracy: {cv_scores.std():.4f}")

In [ ]:

# Define the parameter grid for Multinomial Naive Bayes
param_grid = {
    'alpha': [0.1, 0.5, 1.0, 2.0, 5.0] # Smoothing parameter
}

# Initialize a fresh Multinomial Naive Bayes classifier for tuning
mnb_tuned_model = MultinomialNB()

# Set up GridSearchCV
# We'll use the same stratified 5-fold cross-validation (skf) as before
grid_search = GridSearchCV(
    estimator=mnb_tuned_model,
    param_grid=param_grid,
    cv=skf, # Using the StratifiedKFold instance defined previously
    scoring='accuracy',
    n_jobs=-1, # Use all available CPU cores
    verbose=1
)

# Fit GridSearchCV to the training data (X and y are already defined as X_train and y_train)
grid_search.fit(X, y)

# Print the best parameters and best score
print("\nBest parameters found:", grid_search.best_params_)
print("Best cross-validation accuracy:", grid_search.best_score_)

In [ ]:
# Evaluate the best model on the test set
best_mnb_model = grid_search.best_estimator_
y_pred_tuned_mnb = best_mnb_model.predict(X_test)

print(f"\nTunned Multinomial Naive Bayes Accuracy on Test Set: {accuracy_score(y_test, y_pred_tuned_mnb):.4f}\n")
print("Classification Report for Tunned Multinomial Naive Bayes (Test Set):")
print(classification_report(y_test, y_pred_tuned_mnb))

In [ ]:

print("\n--- Performance Comparison (Test Set) ---")

# Original Multinomial Naive Bayes Model
accuracy_mnb = accuracy_score(y_test, y_pred_mnb)
macro_f1_mnb = f1_score(y_test, y_pred_mnb, average='macro')
print(f"\nOriginal Multinomial Naive Bayes:")
print(f"  Accuracy: {accuracy_mnb:.4f}")
print(f"  Macro F1: {macro_f1_mnb:.4f}")

# Tuned Multinomial Naive Bayes Model
accuracy_tuned_mnb = accuracy_score(y_test, y_pred_tuned_mnb)
macro_f1_tuned_mnb = f1_score(y_test, y_pred_tuned_mnb, average='macro')
print(f"\nTuned Multinomial Naive Bayes (Best Alpha: {grid_search.best_params_['alpha']}):")
print(f"  Accuracy: {accuracy_tuned_mnb:.4f}")
print(f"  Macro F1: {macro_f1_tuned_mnb:.4f}")

# Optionally, display full classification reports again for detailed comparison
print("\n--- Detailed Classification Reports ---")
print("\nClassification Report for Original Multinomial Naive Bayes:")
print(classification_report(y_test, y_pred_mnb))
print("\nClassification Report for Tuned Multinomial Naive Bayes:")
print(classification_report(y_test, y_pred_tuned_mnb))

In [ ]:

# Confusion Matrix for Original Multinomial Naive Bayes
cm_mnb = confusion_matrix(y_test, y_pred_mnb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_mnb, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title('Confusion Matrix: Original Multinomial Naive Bayes')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Confusion Matrix for Tuned Multinomial Naive Bayes
cm_tuned_mnb = confusion_matrix(y_test, y_pred_tuned_mnb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_tuned_mnb, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title('Confusion Matrix: Tuned Multinomial Naive Bayes')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()